In [ ]:
!pip install --quiet datasets transformers peft ipython numpy matplotlib evaluate jiwer librosa tensorboard

In [ ]:
from datasets import load_dataset
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio
from scipy.signal import resample
import torch
from tqdm import tqdm
from torch.utils.data import DataLoader
from transformers import WhisperTokenizer, WhisperFeatureExtractor, WhisperForConditionalGeneration
import evaluate
import pandas as pd
from datasets import Dataset, Audio
import os
from peft import LoraConfig, get_peft_model
import librosa
from torch.utils.tensorboard import SummaryWriter 
import datetime 

In [ ]:
def down_sample_audio(audio_original, original_sample_rate):
    target_sample_rate = 16000
    num_samples = int(len(audio_original) * target_sample_rate / original_sample_rate)
    downsampled_audio = resample(audio_original, num_samples)
    return downsampled_audio

tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-small", language='ar', task='transcribe')
feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-small", language='ar', task='transcribe')

In [ ]:
# --- Model Initialization for New Training ---
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to('cuda')
model.gradient_checkpointing_enable()
model.config.use_cache = False

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

In [ ]:
model = get_peft_model(model, lora_config)
print('LoRA config and model PEFT wrapped done.')

# Verify trainable parameters for the fresh run
print("\n--- Trainable parameters for NEW LoRA model ---")
model.print_trainable_parameters()
print("-------------------------------------------\n")

In [ ]:
#Data Loading
KAGGLE_INPUT_ROOT = "/kaggle/input"
CSV_PATH = os.path.join(KAGGLE_INPUT_ROOT, "linto-dataset", "linto_dataset", "linto_segmented_dataset.csv")
path_updated = os.path.join(KAGGLE_INPUT_ROOT, "linto-dataset", "linto_dataset")

def create_cleaned_audio_path(relative_path, base_dir):
    relative_path_normalized = relative_path.replace("\\", "/")
    full_path_with_original_ext = os.path.join(base_dir, relative_path_normalized)
    root, ext = os.path.splitext(full_path_with_original_ext)
    cleaned_full_path = root + ".wav"
    return os.path.normpath(cleaned_full_path)

print(f"Loading data from: {CSV_PATH}")
try:
    small_df = pd.read_csv(CSV_PATH)
except FileNotFoundError:
    print(f"Error: CSV file not found at {CSV_PATH}.")
    print("Please ensure your dataset is correctly linked and path is accurate.")
    exit()

small_df['audio_path'] = small_df['wav'].apply(lambda x: create_cleaned_audio_path(x, path_updated))

print(f"Example of a constructed audio path: {small_df.iloc[0]['audio_path']}")

In [ ]:
print("\nVerifying audio file existence and filtering dataset...")
valid_rows = []
skipped_audio_count = 0

for index, row in tqdm(small_df.iterrows(), total=len(small_df), desc="Checking audio files"):
    audio_full_path = row['audio_path']
    audio_id = os.path.splitext(os.path.basename(audio_full_path))[0] # Extract audio_id from filename for logging
    if os.path.exists(audio_full_path):
        valid_rows.append(row)
    else:
        skipped_audio_count += 1

small_df2 = pd.DataFrame(valid_rows)
if skipped_audio_count > 0:
    print(f"\nTotal {skipped_audio_count} audio files were skipped due to not being found.")
    print(f"Proceeding with {len(small_df2)} valid entries in the DataFrame.")
else:
    print("\nAll constructed audio paths found. No files skipped during initial verification.")

In [ ]:
data_list_for_hf_dataset = []

for index, row in small_df2.iterrows():
    audio_id_from_path = os.path.splitext(os.path.basename(row['audio_path']))[0]
    data_list_for_hf_dataset.append({
        "audio": {"path": row['audio_path']},
        "sentence": row['wrd'],
        "audio_id": audio_id_from_path
    })

custom_hf_dataset = Dataset.from_list(data_list_for_hf_dataset)
custom_hf_dataset = custom_hf_dataset.cast_column("audio", Audio(sampling_rate=16000)) # Explicitly set sampling_rate

split_custom_dataset = custom_hf_dataset.train_test_split(test_size=0.2, seed=42)

train_data = split_custom_dataset['train']
test_data = split_custom_dataset['test']

print(f"Loaded {len(train_data)} training examples and {len(test_data)} test examples from your local dataset for testing.")

In [ ]:
list_of_transcription_lengths = []
tokenized_text = tokenizer(train_data['sentence']).input_ids
for text in tokenized_text:
    list_of_transcription_lengths.append(len(text))

plt.hist(list_of_transcription_lengths)
plt.xlabel("sentence length")
plt.ylabel("number of transcripts")
plt.title("Distribution of Tokenized Transcription Lengths")
plt.show()

In [ ]:
# Setting max_len based on the histogram analysis
MAX_SEQ_LEN = 140

class whisper_training_dataset(torch.utils.data.Dataset):
    def __init__(self, dataset, max_len):
        self.dataset = dataset
        self.max_len = max_len
        self.bos_token = model.config.decoder_start_token_id

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        try:
            item = self.dataset[idx]
            
            if 'audio' not in item or item['audio'] is None or 'array' not in item['audio'] or 'sampling_rate' not in item['audio']:
                print(f"Skipping item {idx} due to incomplete audio data after loading (should not happen after pre-filtering).")
                return None
            
            audio_data = down_sample_audio(item['audio']["array"], item['audio']["sampling_rate"])
            
            features = feature_extractor(raw_speech=audio_data, sampling_rate=16000, return_tensors='pt')
            input_features = features.input_features[0]
            
            input_attention_mask = features.attention_mask[0] if 'attention_mask' in features and features.attention_mask is not None else None    

            transcription = item["sentence"]

            labels = tokenizer(transcription, padding="max_length", max_length=self.max_len, truncation=True, return_tensors="pt")
            labels = labels["input_ids"].masked_fill(labels['attention_mask'].ne(1), -100)
            labels = labels[0][1:]

            return {
                "input_features": input_features,
                "input_attention_mask": input_attention_mask, # Return the attention mask
                "labels": labels
            }
        except Exception as e:
            print(f"Skipping item {idx} due to an unexpected error during processing: {e}")
            return None

In [ ]:
def collate_fn(batch):
    batch = [item for item in batch if item is not None]
    if not batch:
        return {}

    input_features = torch.stack([x['input_features'] for x in batch])
    labels = torch.stack([x['labels'] for x in batch])
    
    if batch[0]["input_attention_mask"] is not None:
        input_attention_mask = torch.stack([x['input_attention_mask'] for x in batch])
    else:
        input_attention_mask = None 
   
    return {"input_features": input_features, "input_attention_mask": input_attention_mask, "labels": labels} 

train_whisper_dataset = whisper_training_dataset(dataset=train_data, max_len=MAX_SEQ_LEN)

train_dataloader = torch.utils.data.DataLoader(
    train_whisper_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn
)

In [ ]:
from jiwer import wer

def evaluation(model):
    device='cuda'
    
    test_dataset = whisper_training_dataset(dataset=test_data, max_len=MAX_SEQ_LEN)
    test_dataloader = torch.utils.data.DataLoader(
        test_dataset,
        batch_size=8,
        shuffle=False,
        collate_fn=collate_fn,
    )
    
    model.eval()
    predictions=[]
    references=[]

    for batch in tqdm(test_dataloader,total=len(test_dataloader)):
        input_features = batch["input_features"].to(device)
        labels = batch["labels"].to(device)
        
        input_attention_mask = batch["input_attention_mask"] 
        if input_attention_mask is not None:
            input_attention_mask = input_attention_mask.to(device)

        with torch.no_grad():
            generated_tokens = model.generate(
                input_features=input_features,
                attention_mask=input_attention_mask, # Pass the attention mask here
                language='arabic',
                task='transcribe',
                max_new_tokens=MAX_SEQ_LEN
            )

        decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        labels = labels.cpu().numpy()
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        predictions.extend(decoded_preds)
        references.extend(decoded_labels)
        
    WER = wer(references, predictions) * 100
    return WER

In [ ]:
# TensorBoard setup
total_batches_per_epoch = len(train_dataloader)
KAGGLE_WORKING_DIR = "/kaggle/working"
current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
TENSORBOARD_LOG_DIR = os.path.join(KAGGLE_WORKING_DIR, 'runs', current_time)
os.makedirs(TENSORBOARD_LOG_DIR, exist_ok=True)
writer = SummaryWriter(log_dir=TENSORBOARD_LOG_DIR)
print(f"TensorBoard logs will be saved to: {TENSORBOARD_LOG_DIR}")

torch.cuda.empty_cache()
model.train()
device='cuda'
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

max_epochs = 20
running_loss = [] # This now stores *all* step losses
epoch_metrics = [] # To store epoch-level metrics for CSV
running_wer_per_epoch = [] # This now stores WER per epoch for plotting

# --- Early Stopping Parameters ---
best_cer = float('inf') # Initialize with infinity
patience_counter = 0
patience = 5

print("Starting LoRA fine-tuning...")
os.makedirs("/kaggle/working/losses", exist_ok=True)
os.makedirs("/kaggle/working/lora_checkpoints", exist_ok=True)

# Main training loop
for epoch in range(max_epochs): # Start from epoch 0 for new training
    print(f"\nEpoch {epoch + 1}/{max_epochs}")
    
    model.train() # Ensure training mode at the beginning of each epoch
    
    epoch_train_losses = [] # Reset for each epoch to calculate average

    for batch_idx, batch in enumerate(tqdm(train_dataloader, total=total_batches_per_epoch, leave=False, desc=f"Epoch {epoch + 1}")):
        if not batch:
            continue

        input_features, labels = batch["input_features"].to(device), batch["labels"].to(device)

        outputs = model(input_features, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        running_loss.append(loss.item())
        epoch_train_losses.append(loss.item()) # Store for current epoch's average

        global_step = epoch * total_batches_per_epoch + batch_idx
        writer.add_scalar('Loss/Train_Step_Loss', loss.item(), global_step)

        if (batch_idx + 1) % 2500 == 0:
            plt.figure(figsize=(10, 4))
            plt.plot(running_loss)
            plt.xlabel('Steps')
            plt.ylabel('Loss')
            plt.title("Training Loss Over Steps")
            plt.grid(True)
            plt.savefig(f'/kaggle/working/losses/loss_plot_epoch_{epoch+1}_step_{batch_idx+1}.png')
            plt.close() # Close plot to free memory
            print(f"Epoch {epoch + 1}, Step {batch_idx + 1}/{len(train_dataloader)}, Loss: {loss.item():.4f}")

        if (batch_idx + 1) % 2500 == 0:
            save_path = f'/kaggle/working/lora_checkpoints/lora_model_epoch_{epoch+1}_step_{batch_idx+1}'
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            model.save_pretrained(save_path)
            # Optionally save optimizer state at intermediate checkpoints too
            # torch.save(optimizer.state_dict(), os.path.join(save_path, "optimizer_state.pt"))
            print(f"LoRA adapters saved to '{save_path}'")

    torch.cuda.empty_cache()

    current_cer = evaluation(model)
    running_wer_per_epoch.append(current_cer)
    print(f"End of Epoch {epoch + 1}, Validation CER: {current_cer:.2f}%")

    writer.add_scalar('WER/Validation_Epoch_WER', current_cer, epoch)
    
    # Calculate average training loss for the current epoch
    avg_epoch_train_loss = np.mean(epoch_train_losses) if epoch_train_losses else 0
    writer.add_scalar('Loss/Train_Epoch_Loss', avg_epoch_train_loss, epoch) # Log average epoch loss

    # Store epoch metrics
    epoch_metrics.append({
        'epoch': epoch + 1,
        'avg_training_loss': avg_epoch_train_loss,
        'validation_wer': current_cer
    })

    if current_cer < best_cer:
        best_cer = current_cer
        patience_counter = 0
        best_model_save_path = '/kaggle/working/best_lora_model'
        os.makedirs(best_model_save_path, exist_ok=True) # Ensure directory exists
        
        # Save the best model (LoRA adapters)
        model.save_pretrained(best_model_save_path)
        
        # Save the optimizer state ONLY with the best model
        torch.save(optimizer.state_dict(), os.path.join(best_model_save_path, "optimizer_state.pt"))
        
        print(f"New best model and optimizer state saved to '{best_model_save_path}' with WER: {best_cer:.2f}")
    else:
        patience_counter += 1
        print(f"Validation WER did not improve. Patience: {patience_counter}/{patience}")

    if patience_counter >= patience:
        print(f"Early stopping triggered! No improvement for {patience} consecutive epochs.")
        break

writer.close()
print("\nLoRA Fine-tuning complete!")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir=TENSORBOARD_LOG_DIR

In [ ]:
# --- Save epoch metrics to CSV ---
metrics_df = pd.DataFrame(epoch_metrics)
metrics_csv_path = os.path.join(KAGGLE_WORKING_DIR, 'training_metrics.csv')
metrics_df.to_csv(metrics_csv_path, index=False)
print(f"Training metrics saved to: {metrics_csv_path}")

# --- Final Plots ---
# Plot Validation CER over Epochs
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(running_wer_per_epoch) + 1), running_wer_per_epoch, marker='o', linestyle='-')
plt.xlabel('Epoch')
plt.ylabel('Validation WER')
plt.title('Validation WER Over Epochs')
plt.grid(True)
plt.show()

print("\nWER values per epoch (from start of this run):", running_wer_per_epoch)

# --- Instructions for TensorBoard Access ---
print("\nTo access TensorBoard:")
print("1. Go to the 'Output' tab of this notebook after it completes.")
print("2. Scroll down to the 'Logs' section.")
print("3. Click on the 'TensorBoard' button/link. This will launch a new tab with your TensorBoard dashboard.")
print(f"Your logs are located at: {TENSORBOARD_LOG_DIR}")